# 🎬 Script → Video (free, runs on Google Colab)

Turns a text script into a narrated stock-footage video (10+ minutes is fine), with captions and optional music.
Your scripts and finished videos live in **Google Drive → `MyDrive/script2video/`**.

**First time only:**
1. Get a free **Pexels** API key: https://www.pexels.com/api/ (Pixabay is optional: https://pixabay.com/api/docs/).
2. In Colab, click the 🔑 **Secrets** icon in the left sidebar. Add `PEXELS_API_KEY` (and `PIXABAY_API_KEY` if you have one), and switch on "Notebook access".

Then run the cells from top to bottom. Use **Runtime → Run all**, or press ▶ on each cell.

## 1. Connect Google Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
BASE = pathlib.Path('/content/drive/MyDrive/script2video')
for sub in ['scripts', 'outputs', 'footage', 'music']:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

!rm -rf /content/script2video && git clone -q https://github.com/Farhan-Team/script2video /content/script2video
!pip install -q edge-tts gradio requests
!which ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)

# API keys from Colab Secrets (🔑 icon on the left)
from google.colab import userdata
for k in ['PEXELS_API_KEY', 'PIXABAY_API_KEY', 'UNSPLASH_ACCESS_KEY']:
    try:
        os.environ[k] = userdata.get(k) or ''
    except Exception:
        os.environ[k] = ''
print('Keys found:', [k for k in ['PEXELS_API_KEY','PIXABAY_API_KEY','UNSPLASH_ACCESS_KEY'] if os.environ[k]] or 'NONE (visuals will be plain backgrounds)')
print('Put scripts in:', BASE / 'scripts')

## 2a. Option A: web page with a public Cloudflare link
Run this cell, then open the `https://….trycloudflare.com` link it prints.
On that page, upload a script and click **Generate**. Finished videos are also saved to `MyDrive/script2video/outputs`.
The link works for as long as this cell keeps running.

In [ ]:
![ -f /content/cloudflared ] || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared)
import subprocess, time, re, os
env = dict(os.environ, S2V_HEADLESS='1', S2V_OUTPUT_DIR=str(BASE/'outputs'), S2V_WORK_DIR='/content/work',
           S2V_MEDIA_DIR=str(BASE/'footage'))
app = subprocess.Popen(['python', 'app.py'], cwd='/content/script2video', env=env,
                       stdout=open('/content/app.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(8)
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('\n👉 Open your app here:', m.group(0), '\n')
        break
try:
    tun.wait()            # keep running; stop the cell to shut down
except KeyboardInterrupt:
    tun.terminate(); app.terminate()

## 2b. Option B: render every script in Drive (no web page)
Put `.txt` or `.md` scripts in `MyDrive/script2video/scripts`. Optionally add `bg.mp3` to the `music` folder,
and your own clips to `footage` (they're matched by file name).
Run this cell. Each video is saved to `outputs`, and scripts that already have a video are skipped.

In [ ]:
VOICE  = 'en-US-GuyNeural'     # e.g. en-US-JennyNeural, en-GB-RyanNeural, ur-PK-AsadNeural
FORMAT = ''                    # '' = 1920x1080,  '--vertical' = Shorts,  '--size 1280x720' = faster
MUSIC  = BASE / 'music' / 'bg.mp3'

import shlex
for script in sorted((BASE/'scripts').glob('*.[tm][xd]*')):
    out = BASE / 'outputs' / (script.stem + '.mp4')
    if out.exists():
        print('skip (already done):', out.name); continue
    music = f'--music {shlex.quote(str(MUSIC))}' if MUSIC.exists() else ''
    print('▶', script.name)
    !cd /content/script2video && python script2video.py {shlex.quote(str(script))} -o {shlex.quote(str(out))} \
        --workdir /content/work/{shlex.quote(script.stem)} --voice {VOICE} {FORMAT} {music} \
        --media-dir {shlex.quote(str(BASE/'footage'))} --workers 2

**Tips**
- 150 words of script ≈ 1 minute, so 1,500 words ≈ 10 minutes.
- To choose the footage for a paragraph yourself, add a tag like `[search: city skyline at night]`. The tag is not read aloud.
- If Colab disconnects, run the cells again. Finished parts are cached for that session.